# GlobalPref — Quick Rapidata Demo

A minimal demo comparing **GPT-5** vs **Claude Sonnet 4.6** using [Rapidata](https://rapidata.ai) human feedback.

**Pipeline:** 5 prompts → generate response pairs → submit to Rapidata → collect votes → show results.

## 1. Setup & Imports

In [1]:
# Install dependencies
!pip install -q openai anthropic rapidata python-dotenv

import json
import os
import random
from datetime import datetime
from rapidata import RapidataClient
from dotenv import load_dotenv

from openai import OpenAI
from anthropic import Anthropic


# --- Helpers ---

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

# --- Load API keys from .env file ---

load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env file"
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env file"

log("Setup complete.")

ERROR: Ignored the following versions that require a different python version: 0.1.0 Requires-Python <4.0,>=3.12; 0.1.10 Requires-Python <4.0,>=3.12; 0.1.11 Requires-Python <4.0,>=3.12; 0.1.12 Requires-Python <4.0,>=3.12; 0.1.13 Requires-Python <4.0,>=3.12; 0.1.14 Requires-Python <4.0,>=3.12; 0.1.15 Requires-Python <4.0,>=3.12; 0.1.16 Requires-Python <4.0,>=3.12; 0.1.17 Requires-Python <4.0,>=3.12; 0.1.18 Requires-Python <4.0,>=3.10; 0.1.19 Requires-Python <4.0,>=3.10; 0.1.20 Requires-Python <4.0,>=3.10; 0.1.21 Requires-Python <4.0,>=3.10; 0.1.22 Requires-Python <4.0,>=3.10; 0.1.23 Requires-Python <4.0,>=3.10; 0.1.24 Requires-Python <4.0,>=3.10; 0.1.25 Requires-Python <4.0,>=3.10; 0.1.3 Requires-Python <4.0,>=3.12; 0.1.5 Requires-Python <4.0,>=3.12; 0.1.6 Requires-Python <4.0,>=3.12; 0.1.7 Requires-Python <4.0,>=3.12; 0.1.8 Requires-Python <4.0,>=3.12; 0.1.9 Requires-Python <4.0,>=3.12; 0.2.0 Requires-Python <4.0,>=3.10; 0.2.1 Requires-Python <4.0,>=3.10; 0.3.0 Requires-Python <4.0,>=3

ModuleNotFoundError: No module named 'rapidata'

## 2. Generate Prompts

5 prompts (one per category) to keep this quick.

In [ ]:
prompts = [
    {"prompt_id": "factual_001",        "category": "factual",          "prompt": "What causes inflation?"},
    {"prompt_id": "practical_001",      "category": "practical_advice", "prompt": "How should I negotiate a salary?"},
    {"prompt_id": "creative_001",       "category": "creative",         "prompt": "Write a short poem about rain."},
    {"prompt_id": "opinion_001",        "category": "opinion_values",   "prompt": "What makes a good leader?"},
    {"prompt_id": "everyday_001",       "category": "everyday_tasks",   "prompt": "How do I apologize to a friend after a disagreement?"},
]

save_json(prompts, "prompts.json")
log(f"Saved {len(prompts)} prompts to prompts.json")

## 3. Generate Response Pairs

Call GPT-5 and Claude Sonnet 4.6 for each prompt. A/B position is randomized per prompt to eliminate position bias.

In [ ]:
SYSTEM_PROMPT = "You are a helpful assistant."
MAX_TOKENS = 500
OPENAI_MODEL = "gpt-5"
ANTHROPIC_MODEL = "claude-sonnet-4-6"

openai_client = OpenAI()
anthropic_client = Anthropic()
rng = random.Random(42)

prompts = load_json("prompts.json")
pairs = []

for i, prompt_data in enumerate(prompts):
    text = prompt_data["prompt"]
    log(f"[{i+1}/{len(prompts)}] {prompt_data['prompt_id']}: {text}")

    gpt_response = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        max_completion_tokens=MAX_TOKENS,
    ).choices[0].message.content

    claude_response = anthropic_client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=MAX_TOKENS,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": text}],
    ).content[0].text

    # Randomize position to eliminate bias
    if rng.random() < 0.5:
        response_a, response_b = gpt_response, claude_response
        model_a, model_b = "gpt-5", "claude-sonnet-4-6"
    else:
        response_a, response_b = claude_response, gpt_response
        model_a, model_b = "claude-sonnet-4-6", "gpt-5"

    pairs.append({
        "prompt_id": prompt_data["prompt_id"],
        "category": prompt_data["category"],
        "prompt": text,
        "response_a": response_a,
        "response_b": response_b,
        "model_a": model_a,
        "model_b": model_b,
    })

save_json(pairs, "response_pairs.json")
log(f"Saved {len(pairs)} response pairs to response_pairs.json")

## 4. Rapidata

Submit response pairs as a compare job to the global Rapidata audience. Each step below mirrors the [Rapidata quickstart](https://docs.rapidata.ai/latest/quickstart/).

In [ ]:
RESPONSES_PER_DATAPOINT = 15
INSTRUCTION = "Which response is more helpful and accurate?"
ORDER_ID_PATH = "order_id.json"

pairs = load_json("response_pairs.json")

datapoints = [[p["response_a"], p["response_b"]] for p in pairs]
contexts = [p["prompt"] for p in pairs]

log(f"Prepared {len(datapoints)} datapoints for Rapidata.")

In [ ]:
### Step 1: Get an Audience

In [ ]:
client = RapidataClient()
audience = client.audience.get_audience_by_id("global")
log("Connected to global audience.")

### Step 2: Create Job Definition

In [ ]:
job_definition = client.job.create_compare_job_definition(
    name="GlobalPref",
    instruction=INSTRUCTION,
    datapoints=datapoints,
    contexts=contexts,
    data_type="text",
    responses_per_datapoint=RESPONSES_PER_DATAPOINT,
)

log("Job definition created.")

### Step 3: Preview

In [ ]:
job_definition.preview()

### Step 4: Submit Job

In [ ]:
if os.path.exists(ORDER_ID_PATH):
    order_info = load_json(ORDER_ID_PATH)
    log(f"Order already exists: {order_info['order_id']}")
else:
    job = audience.assign_job(job_definition)
    job_id = getattr(job, 'job_id', None) or getattr(job, 'id', None) or str(job)
    order_info = {"order_id": job_id, "created_at": datetime.now().isoformat()}
    save_json(order_info, ORDER_ID_PATH)
    log(f"Order created: {job_id}")

print(f"Order ID: {order_info['order_id']}")

## 5. Collect Results

### Monitor Progress

In [ ]:
order_info = load_json("order_id.json")
job = client.job.get_job_by_id(order_info["order_id"])
job.display_progress_bar()

### Get Results

In [ ]:
results = job.get_results()
save_json(results, "results_raw.json")

log("Results saved to results_raw.json")
n_results = len(results.get("results", []))
print(f"  {n_results} datapoint results collected")

## 6. Build Dataset

Merge response pairs with Rapidata results. One row per prompt with total vote counts.

In [ ]:
DATASET_PATH = "dataset.jsonl"

pairs = load_json("response_pairs.json")
results_raw = load_json("results_raw.json")

prompt_to_pair = {p["prompt"]: p for p in pairs}
dataset = []

for result in results_raw.get("results", []):
    context = result.get("context", "")
    pair = prompt_to_pair.get(context)
    if not pair:
        log(f"WARNING: no matching prompt for context: {context[:80]}...")
        continue

    votes_a, votes_b = 0, 0
    for vote in result.get("detailedResults", []):
        voted_for = vote.get("votedFor", "")
        if voted_for == pair["response_a"]:
            votes_a += 1
        elif voted_for == pair["response_b"]:
            votes_b += 1

    total = votes_a + votes_b
    if total == 0:
        continue

    preferred = "a" if votes_a > votes_b else ("b" if votes_b > votes_a else "tie")

    dataset.append({
        "prompt_id": pair["prompt_id"],
        "category": pair["category"],
        "prompt": pair["prompt"],
        "model_a": pair["model_a"],
        "model_b": pair["model_b"],
        "votes_a": votes_a,
        "votes_b": votes_b,
        "total_votes": total,
        "preferred": preferred,
        "win_rate_a": round(votes_a / total, 4),
    })

with open(DATASET_PATH, "w", encoding="utf-8") as f:
    for row in dataset:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

log(f"Saved {len(dataset)} rows to {DATASET_PATH}")

for row in dataset:
    winner = f"{row['model_a']} wins" if row["preferred"] == "a" else (
        f"{row['model_b']} wins" if row["preferred"] == "b" else "tie")
    print(f"  {row['prompt_id']}: {row['votes_a']}-{row['votes_b']} → {winner}")